In [5]:
import pandas as pd
import numpy as np

d1 = pd.read_csv("dataset1_site_night.csv", parse_dates=["night_date"])
d2 = pd.read_csv("dataset2_gan_observations.csv", parse_dates=["night_date", "obs_local_time"])

print("Dataset 1:", d1.shape)
print("Dataset 2:", d2.shape)


Dataset 1: (46768, 28)
Dataset 2: (3579, 38)


In [ ]:
#Keeping the matched reports only, 695 reports were more than 25 km from any site, so they have no weather, moon or smoke data attached and can't be used for modelling.
d2 = d2[d2["matched"]].copy()
print("A0 matched only:", len(d2))

A0 matched only: 2884


In [7]:
#Dropping dup submissions, 234 reports have the same time, location and rating as another report, just a different ID. They're double submissions. Keeping them would count the same observation twice and give those nights extra weight.
dup_cols = ["obs_local_time", "obs_latitude", "obs_longitude", "limiting_mag"]
before = len(d2)
d2 = d2.drop_duplicates(subset=dup_cols, keep="first")
print(f"A1 removed {before - len(d2)} duplicates -> {len(d2)} rows")

A1 removed 234 duplicates -> 2650 rows


In [8]:
#Drop rows with no limiting magnitude
before = len(d2)
d2 = d2.dropna(subset=["limiting_mag"])
print(f"A2 removed {before - len(d2)} rows -> {len(d2)} rows")

A2 removed 170 rows -> 2480 rows


In [9]:
#Drop daytime timestamps
before = len(d2)
daytime = d2["obs_local_time"].dt.hour.between(6, 17)
d2 = d2[~daytime]
print(f"A3 removed {before - len(d2)} daytime reports -> {len(d2)} rows")

A3 removed 306 daytime reports -> 2174 rows


In [10]:
#Dropping rows where limiting_mag = 0, as these are likely to be invalid reports.
before = len(d2)
d2 = d2[d2["limiting_mag"] > 0]
print(f"A4 removed {before - len(d2)} rows -> {len(d2)} rows")

A4 removed 86 rows -> 2088 rows


In [11]:
#Dropping rows before 2018 in Dataset 1, there's one stray row per site dated 2017-12-31, before the study period, with no smoke or radiance data. Dropping them also removes the only missing smoke and radiance values.
before = len(d1)
d1 = d1[d1["night_date"] >= "2018-01-01"].copy()
print(f"B1 removed {before - len(d1)} rows -> {len(d1)} rows")
print("Missing smoke:", d1["hms_smoke_level"].isna().sum(),
      " Missing radiance:", d1["viirs_radiance"].isna().sum())


B1 removed 16 rows -> 46752 rows
Missing smoke: 0  Missing radiance: 0


In [13]:
#Imputing a blank in PM2.5 for where the monitor is more than 50 km away, a monitor more than 50 km away doesn't represent air quality at the site. This affects RMNP (~59 km) and Jackson Lake (~52 km). The monitor ID and distance are kept so the change can be traced.
for df in (d1, d2):
    far = df["pm25_monitor_km"] > 50
    df.loc[far, "pm25_ugm3"] = np.nan
    print("PM2.5 blanked PM2.5 on", far.sum(), "rows")

PM2.5 blanked PM2.5 on 5601 rows
PM2.5 blanked PM2.5 on 25 rows


In [14]:
# Adding a missing flag for rows that dont have PM2.5 data
for df in (d1, d2):
    df["pm25_missing"] = df["pm25_ugm3"].isna().astype(int)

print("B4 flagged — D1:", d1["pm25_missing"].sum(), " D2:", d2["pm25_missing"].sum())

B4 flagged — D1: 9970  D2: 138


In [16]:
#Getting rid of spm_readings that are not between the range of 14 and 23
bad_sqm = (d2["sqm_reading"] < 14) | (d2["sqm_reading"] > 23)
d2.loc[bad_sqm, "sqm_reading"] = np.nan
print(f"Blanked {bad_sqm.sum()} readings -> {d2['sqm_reading'].notna().sum()} left")

Blanked 0 readings -> 327 left


In [17]:
#Filling in missing constellation with "Unknown", as the missing values are likely due to the observer not knowing the constellation, rather than the constellation being unknown.
print("B8 filling", d2["constellation"].isna().sum(), "rows")
d2["constellation"] = d2["constellation"].fillna("unknown")

B8 filling 9 rows


In [18]:
#Setting negative PM2.5 to 0
for df in (d1, d2):
    print("C3 negatives:", (df["pm25_ugm3"] < 0).sum())
    df["pm25_ugm3"] = df["pm25_ugm3"].clip(lower=0)

C3 negatives: 0
C3 negatives: 0


In [19]:
#Log transformation of PM2.5, radiance, and precipitation, as they are heavily skewed
for df in (d1, d2):
    for col in ["precip_mm_sum", "pm25_ugm3", "viirs_radiance"]:
        df["log_" + col] = np.log1p(df[col])

d2[["log_precip_mm_sum", "log_pm25_ugm3", "log_viirs_radiance"]].describe().round(2)

,log_precip_mm_sum,log_pm25_ugm3,log_viirs_radiance
count,2088.00,1950.00,2088.00
mean,0.14,1.90,3.20
std,0.41,0.63,1.61
min,0.00,0.18,0.00
25%,0.00,1.46,1.79
50%,0.00,1.81,3.86
75%,0.00,2.34,4.19
max,2.61,3.77,5.28


In [21]:
#Dropping a redundant PM2.5 column, it is just a copy of the original but it if for a day later
d1 = d1.drop(columns=["pm25_next_day_ugm3"])
d2 = d2.drop(columns=["pm25_next_day_ugm3"])
print("D1:", d1.shape, " D2:", d2.shape)

KeyError: "['pm25_next_day_ugm3'] not found in axis"

In [ ]:
d1.to_csv("dataset1_site_night_cleaned.csv", index=False)
d2.to_csv("dataset2_gan_observations_cleaned.csv", index=False)

Saved.
